# Исследование для GitHub Pages
Запустите установку, затем исследование. До сбора вы проверите профиль и вручную утвердите названия и сайты конкурентов. Автопоиск использует TAVILY_API_KEY из Secrets только после вашего выбора; один basic-запрос, без повторов.
Сбор — до 2 страниц на сайт. LLM здесь не вызывается. Результат product-result.json читается на Pages локально, не публикуется автоматически.
Если Colab запросит доступ к секрету для этого блокнота, разрешите его. Секреты привязаны к аккаунту, доступ — отдельно к каждому блокноту.

In [ ]:
# Установка или обновление кода. Секреты не используются.
from pathlib import Path
import subprocess, sys, os
work = Path("/content/neuro-analyst")
remote = "https://github.com/gorlanova-automation/competitor-neuro-analyst.git"
if work.exists():
    actual = subprocess.check_output(["git", "-C", str(work), "remote", "get-url", "origin"], text=True).strip()
    assert actual == remote, "В этой папке другой проект; не изменяем его."
    subprocess.check_call(["git", "-C", str(work), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", remote, str(work)])
os.chdir(work)
if str(work) not in sys.path: sys.path.insert(0, str(work))
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-r", "app/requirements.txt", "-r", "stage_3/code/requirements.txt"])
print("Установка завершена.")


In [ ]:
# Получить секрет до начала input(), чтобы ответы Colab не смешивались.
from google.colab import userdata
_search_key = None
try:
    _search_key = userdata.get("TAVILY_API_KEY")
    print("Секрет подключён. Значение не выводится.")
except Exception:
    print("Секрет недоступен: можно продолжить без автоматического поиска.")


In [ ]:
from app.colab_product import launch
try:
    project = launch(output_dir="/content/neuro-analyst/var/colab_product", search_key=_search_key)
finally:
    _search_key = None
